In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from torch.utils.data import DataLoader
from torch.optim import AdamW
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
import re
from tqdm import tqdm

In [ ]:
MODEL_NAME = "Llama-3.2-1B-Instruct"
LORA_R = 8
LORA_DROPOUT = 0.1
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1
MAX_LENGTH = 512
TEST_SAMPLE_SIZE = 64
TEST_BATCH_SIZE = 8
MAX_NEW_TOKENS = 512
device = torch.device("cuda:0")
SYSTEM_PROMPT = "You are a mathematical problem-solving assistant. Please provide the final answer to the following math problem. Your final answer should be enclosed in \\boxed{}."

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)
tokenizer.padding_side = 'left'

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
PAD_TOKEN_ID = tokenizer.pad_token_ids

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map={"": 0},
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
train_dataset = load_dataset("./data/hendrycks_math", 'algebra', split="train")
test_dataset = load_dataset("./data/hendrycks_math", 'algebra', split="test")

def extract_boxed_answer(text):
    matches = re.findall(r'\\boxed\{([^}]*)\}', text)
    return matches[-1].strip() if matches else None

def normalize_answer(answer):
    return answer.replace(" ", "").lower() if answer else None

def process_train_data(examples):
    texts = []
    answers = []
    
    for problem, solution in zip(examples['problem'], examples['solution']):
        answer = extract_boxed_answer(solution)
        answers.append(normalize_answer(answer))
        
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Problem: {problem}"},
            {"role": "assistant", "content": f"{solution}"}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    
    tokenized = tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding="max_length")
    
    tokenized["labels"] = [
        [-100 if token_id == PAD_TOKEN_ID else token_id for token_id in input_ids]
        for input_ids in tokenized["input_ids"]
    ]
    tokenized["attention_mask"] = [
        [0 if token_id == PAD_TOKEN_ID else 1 for token_id in input_ids]
        for input_ids in tokenized["input_ids"]
    ]
    
    tokenized["answer"] = answers
    tokenized["problem"] = examples['problem']
    
    return tokenized

def process_test_data(examples):
    texts = []
    answers = []
    for problem, solution in zip(examples['problem'], examples['solution']):
        answer = extract_boxed_answer(solution)
        answers.append(normalize_answer(answer))
        
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Problem: {problem}"},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    
    tokenized = tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding="max_length")
    tokenized["labels"] = [
        [-100 if token_id == PAD_TOKEN_ID else token_id for token_id in input_ids]
        for input_ids in tokenized["input_ids"]
    ]
    tokenized["attention_mask"] = [
        [0 if token_id == PAD_TOKEN_ID else 1 for token_id in input_ids]
        for input_ids in tokenized["input_ids"]
    ]

    tokenized["answer"] = answers
    tokenized["problem"] = examples['problem']
    
    return tokenized

tokenized_train_dataset = train_dataset.map(
    process_train_data,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Processing train data"
)
tokenized_test_dataset = test_dataset.map(
    process_test_data, 
    batched=True, 
    remove_columns=test_dataset.column_names
)

In [ ]:
def evaluate_model(model, tokenized_dataset, num_samples=None, batch_size=8):
    test_samples = tokenized_dataset.select(range(min(num_samples or len(tokenized_dataset), len(tokenized_dataset))))
    model.eval()
    
    input_ids_list = test_samples['input_ids']
    attention_mask_list = test_samples['attention_mask']
    true_answers = test_samples['answer']
    
    correct = 0
    total = 0
    
    for i in tqdm(range(0, len(input_ids_list), batch_size), desc="Evaluating"):
        batch_input_ids = torch.tensor(input_ids_list[i:i+batch_size], dtype=torch.long).to(device)
        batch_attention_mask = torch.tensor(attention_mask_list[i:i+batch_size], dtype=torch.long).to(device)
        batch_true_answers = true_answers[i:i+batch_size]
        with torch.no_grad():
            outputs = model.generate(
                input_ids=batch_input_ids,
                attention_mask=batch_attention_mask,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=PAD_TOKEN_ID
            )
        
        for output, true_ans in zip(outputs, batch_true_answers):
            response = tokenizer.decode(output, skip_special_tokens=True)
            pred_ans = extract_boxed_answer(response)
            if normalize_answer(pred_ans) == true_ans:
                correct += 1
            total += 1
    accuracy = correct / total * 100
    print("EVALUATION RESULTS")
    print(f"Accuracy: {accuracy:.2f}% ({correct}/{total})")
    return accuracy, correct, total


In [ ]:
baseline_accuracy, baseline_correct, baseline_total = evaluate_model(
    model, tokenized_test_dataset, num_samples=TEST_SAMPLE_SIZE, batch_size=TEST_BATCH_SIZE
)

In [ ]:
def collate_fn(features):
    return {
        "input_ids": torch.tensor([f["input_ids"] for f in features], dtype=torch.long),
        "attention_mask": torch.tensor([f["attention_mask"] for f in features], dtype=torch.long),
        "labels": torch.tensor([f["labels"] for f in features], dtype=torch.long)
    }

train_dataloader = DataLoader(tokenized_train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)

# 训练循环
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    for step, batch in enumerate(tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")):
        batch = {k: v.to(device) for k, v in batch.items()}
        
        outputs = model(**batch)
        loss = outputs.loss / GRADIENT_ACCUMULATION_STEPS
        
        loss.backward()
        total_loss += loss.item()
        
        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()
            # print(f"Loss {total_loss}")
            total_loss = 0
    
    torch.cuda.empty_cache()

In [ ]:
accuracy, correct, total = evaluate_model(
    model, tokenized_test_dataset, num_samples=TEST_SAMPLE_SIZE, batch_size=TEST_BATCH_SIZE
)